# 🔬 Notebook 2 — Extraction Quality Evaluation
## TeleCopilot-AI — Codebase Extractor Benchmark

This notebook evaluates the quality of every production codebase extractor in `src/ingestion/extractors/`
against the **verified Ground Truth** produced in Notebook 1.

### Pipeline
```
data/raw/ ──► ExtractorFactory ──► ExtractedDocument
                                         │
data/ground_truth/ ──────────────────────┤
                                         ▼
                        Section Alignment (Fuzzy Jaccard)
                                         │
                                         ▼
                  ┌──────────────────────┬──────────────────────┐
                  │  Deterministic       │  Gemini LLM Judge    │
                  │  Metrics             │  (Sliding Window)    │
                  │  Word F1, ROUGE-L    │  4-Dimension Rubric  │
                  │  Entity, Order τ     │  PASS/COND./FAIL     │
                  └──────────┬───────────┴──────────────────────┘
                             ▼
              Human-In-The-Loop (HITL) Review
                             │
                             ▼
   Micro/Macro + Per-Type Aggregation → Reports
```

---

## ⚙️ Section 1 — Setup & Environment

In [ ]:
import sys, os, json, re
from pathlib import Path

# ── Path configuration ──────────────────────────────────────────────────────
# Notebooks live in evals/notebooks/; work from the repo root so data/ paths and `import evals` resolve.
# `telecopilot` itself comes from the installed package (pip install -e .).
PROJECT_ROOT   = Path.cwd().parents[1] if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(PROJECT_ROOT)
RAW_DATA_DIR  = PROJECT_ROOT / 'data' / 'raw'
GT_DIR        = PROJECT_ROOT / 'data' / 'ground_truth'
REPORTS_DIR   = PROJECT_ROOT / 'evals' / 'reports' / 'kb' / 'v1'

assert GT_DIR.exists(), f'❌ Ground Truth directory not found: {GT_DIR}\nRun 01_create_ground_truth.ipynb first!'
print(f'✅ Project root : {PROJECT_ROOT}')
print(f'✅ Ground truth : {GT_DIR}')

In [ ]:
# ── Load .env and initialise Gemini ─────────────────────────────────────────
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY', '')
gemini_client = None
if GEMINI_API_KEY:
    try:
        import google.generativeai as genai
        genai.configure(api_key=GEMINI_API_KEY)
        gemini_client = genai.GenerativeModel('gemini-1.5-flash')
        print(f'✅ Gemini client ready (gemini-1.5-flash)')
    except Exception as e:
        print(f'⚠️  Gemini init failed: {e} — heuristic fallback will be used')
else:
    print('⚠️  No GEMINI_API_KEY — heuristic judge will be used for all evaluations')

# ── Import codebase extractors ───────────────────────────────────────────────
from telecopilot.kb.factories import ExtractorFactory
from telecopilot.kb.extractors.html_extractor import HTMLExtractor
from telecopilot.kb.extractors.xml_extractor import XMLExtractor
from telecopilot.kb.extractors.docx_extractor import DocxExtractor
from telecopilot.kb.extractors.gemini_ocr_extractor import GeminiOCRExtractor
from telecopilot.kb.models import ExtractedDocument
print('✅ Codebase extractors imported')

# ── Import evaluation toolkit ────────────────────────────────────────────────
from evals.extraction.ground_truth_builder import load_ground_truth_md
from evals.extraction.metrics import (
    compute_all_metrics, aggregate_per_type, ExtractionMetrics
)
from evals.extraction.llm_judge import GeminiJudge, JudgeResult
from evals.extraction.hitl import HITLReviewer, save_benchmark_report
print('✅ Evaluation toolkit imported')

judge    = GeminiJudge(gemini_client=gemini_client)
reviewer = HITLReviewer(report_dir=str(REPORTS_DIR))

## 📂 Section 2 — Load Ground Truth Files

In [ ]:
import pandas as pd

gt_files = sorted(GT_DIR.glob('*.md'))
assert len(gt_files) > 0, f'No .md files found in {GT_DIR}'

ground_truths = {}  # stem → (frontmatter, body_text, sections)
for gt_path in gt_files:
    fm, body, sections = load_ground_truth_md(str(gt_path))
    ground_truths[gt_path.stem] = {
        'frontmatter': fm,
        'body': body,
        'sections': sections,
        'section_titles': [s.title for s in sorted(sections, key=lambda s: s.order)],
        'table_cells': fm.get('total_table_cells', 0),
        'media_count': fm.get('total_media_count', 0),
        'format': fm.get('file_format', 'unknown'),
    }

summary_rows = [{
    'Ground Truth File': k + '.md',
    'Format': v['format'],
    'Sections': len(v['sections']),
    'Table Cells (GT)': v['table_cells'],
    'Media (GT)': v['media_count'],
    'GT Words': v['frontmatter'].get('total_words', '—'),
} for k, v in ground_truths.items()]

display(pd.DataFrame(summary_rows).style.set_caption('Loaded Ground Truth Documents'))
print(f'\n✅ Loaded {len(ground_truths)} ground truth documents')

## 🔧 Section 3 — Execute Codebase Extractors

In [ ]:
import traceback, time

SUPPORTED_EXTS = {'.html', '.htm', '.xml', '.docx', '.doc', '.pdf'}
raw_files = sorted([f for f in RAW_DATA_DIR.iterdir() if f.suffix.lower() in SUPPORTED_EXTS])

extracted_docs: dict[str, ExtractedDocument] = {}
extraction_times: dict[str, float] = {}
extraction_errors: dict[str, str] = {}

for raw_file in raw_files:
    print(f'\n⚙️  Extracting: {raw_file.name}')
    t0 = time.perf_counter()
    try:
        extractor = ExtractorFactory.get_extractor(str(raw_file))
        doc = extractor.extract(str(raw_file))
        elapsed = time.perf_counter() - t0
        extracted_docs[raw_file.stem] = doc
        extraction_times[raw_file.stem] = round(elapsed, 3)
        print(f'   ✅ {type(extractor).__name__} → {len(doc.raw_text):,} chars | '
              f'{len(doc.sections)} sections | {elapsed:.3f}s')
    except Exception as e:
        elapsed = time.perf_counter() - t0
        extraction_errors[raw_file.stem] = str(e)
        extraction_times[raw_file.stem] = round(elapsed, 3)
        print(f'   ❌ ERROR: {e}')
        traceback.print_exc()

print(f'\n✅ Extracted {len(extracted_docs)}/{len(raw_files)} documents')
if extraction_errors:
    print(f'⚠️  Errors: {", ".join(extraction_errors.keys())}')

In [ ]:
# Validate ExtractedDocument schema for each extraction
print('Validating ExtractedDocument schema...')
for stem, doc in extracted_docs.items():
    assert doc.document_id, f'{stem}: missing document_id'
    assert doc.raw_text,    f'{stem}: empty raw_text'
    assert doc.sections is not None, f'{stem}: sections is None'
    assert doc.policy_type, f'{stem}: missing policy_type'
    print(f'  ✅ {stem}: doc_id={doc.document_id}, sections={len(doc.sections)}, '
          f'chars={len(doc.raw_text):,}, policy={doc.policy_type}')
print('\n✅ All ExtractedDocument objects are schema-valid')

## 🔗 Section 4 — Section Alignment (Fuzzy Jaccard)

Before computing per-section metrics, we align extracted sections to GT sections using
**Jaccard similarity on section title tokens + first 50 content tokens** (Fix 3).

In [ ]:
from evals.extraction.metrics import tokenize

def jaccard_similarity(text_a: str, text_b: str) -> float:
    """Token-level Jaccard similarity between two text strings."""
    set_a = set(tokenize(text_a))
    set_b = set(tokenize(text_b))
    union = set_a | set_b
    return len(set_a & set_b) / len(union) if union else 0.0

def align_sections(extracted_sections, gt_sections, threshold: float = 0.1):
    """
    For each GT section, find the best-matching extracted section.
    Uses title + first 50 tokens of content.
    Returns list of (gt_section, best_extracted_section_or_None, similarity_score).
    """
    alignments = []
    for gt_sec in gt_sections:
        gt_key = gt_sec.title + ' ' + gt_sec.content[:200]
        best_score, best_match = 0.0, None
        for ex_sec in extracted_sections:
            ex_key = ex_sec.get('section_title', '') + ' ' + ex_sec.get('content', '')[:200]
            score = jaccard_similarity(gt_key, ex_key)
            if score > best_score:
                best_score, best_match = score, ex_sec
        alignments.append((gt_sec, best_match if best_score >= threshold else None, best_score))
    return alignments

# Compute alignments for each document
section_alignments = {}
for stem, doc in extracted_docs.items():
    if stem not in ground_truths:
        continue
    gt = ground_truths[stem]
    alignments = align_sections(doc.sections, gt['sections'])
    matched = sum(1 for _, m, _ in alignments if m is not None)
    section_alignments[stem] = alignments
    print(f'  {stem}: {matched}/{len(gt["sections"])} GT sections aligned '
          f'(threshold=0.10)')

print('\n✅ Section alignment complete')

## 📊 Section 5 — Quantitative Metrics Per File

In [ ]:
all_metrics: list[ExtractionMetrics] = []

for stem, doc in extracted_docs.items():
    if stem not in ground_truths:
        print(f'⚠️  No GT found for {stem} — skipping')
        continue

    gt = ground_truths[stem]
    gt_text = gt['body']
    ex_text = doc.raw_text
    file_type = gt['format']

    # Section titles for order metric
    extracted_titles = [s.get('section_title', '') for s in doc.sections]
    gt_titles = gt['section_titles']

    # Table cells in extracted doc
    extracted_table_cells = sum(
        len(s.get('content', '').split('|'))
        for s in doc.sections
        if '|' in s.get('content', '')
    )

    metrics = compute_all_metrics(
        extracted_text=ex_text,
        ground_truth_text=gt_text,
        file_name=stem,
        file_type=file_type,
        extracted_section_titles=extracted_titles,
        gt_section_titles=gt_titles,
        extracted_table_cells=extracted_table_cells,
        gt_table_cells=gt['table_cells'],
        extracted_media_count=0,   # Current codebase doesn't extract media
        gt_media_count=gt['media_count'],
    )
    all_metrics.append(metrics)

    print(f'\n📄 {stem} ({file_type.upper()})')
    print(f'   Word Recall: {metrics.word_recall:.1%} | Precision: {metrics.word_precision:.1%} | F1: {metrics.word_f1:.1%}')
    print(f'   ROUGE-L:     {metrics.rouge_l_score:.1%}')
    print(f'   Entity Ret.: {metrics.entity_recall_overall:.1%}')
    print(f'   Sec. Order τ:{metrics.section_order_tau:.3f}')
    print(f'   Table Ret.:  {metrics.table_cell_retention:.1%} | Media Ret.: {metrics.media_retention:.1%}')

print(f'\n✅ Metrics computed for {len(all_metrics)} documents')

In [ ]:
# ── Metrics summary table ────────────────────────────────────────────────────
metrics_rows = []
for m in all_metrics:
    metrics_rows.append({
        'File': m.file_name,
        'Type': m.file_type.upper(),
        'Word Recall': f'{m.word_recall:.1%}',
        'Word Precision': f'{m.word_precision:.1%}',
        'Word F1': f'{m.word_f1:.1%}',
        'ROUGE-L': f'{m.rouge_l_score:.1%}',
        'Entity Ret.': f'{m.entity_recall_overall:.1%}',
        'Order τ': f'{m.section_order_tau:.3f}',
        'Table Ret.': f'{m.table_cell_retention:.1%}',
        'Media Ret.': f'{m.media_retention:.1%}',
        'Extr. Time': f'{extraction_times.get(m.file_name, 0):.3f}s',
    })

display(pd.DataFrame(metrics_rows).style.set_caption('Per-File Extraction Metrics'))

## 🤖 Section 6 — Gemini LLM-as-a-Judge

Evaluates each extracted document using the Gemini telecom rubric across 4 dimensions:
1. Completeness (40% weight)
2. Structure Fidelity (25% weight)
3. Noise & Hallucination (20% weight)
4. Arabic / Multilingual Fidelity (15% weight)

In [ ]:
all_judge_results: list[JudgeResult] = []

for stem, doc in extracted_docs.items():
    if stem not in ground_truths:
        continue

    gt = ground_truths[stem]
    file_type = gt['format']
    print(f'\n🧠 Judging: {stem} ({file_type.upper()})', end='')

    # For PDF: pass embedded images for multimodal judging if available
    page_images = None  # Loaded from GT builder if needed

    result = judge.evaluate(
        extracted_text=doc.raw_text,
        ground_truth_text=gt['body'],
        file_name=stem,
        file_type=file_type,
        page_images=page_images,
    )
    all_judge_results.append(result)

    print(f' → {result.verdict_emoji} {result.verdict} (Score: {result.overall_score:.1f}/100)')
    if result.missing_critical_elements:
        print(f'   Missing: {", ".join(result.missing_critical_elements[:3])}')
    if result.recommendations:
        print(f'   Rec: {result.recommendations[0]}')

print(f'\n✅ Judge evaluation complete for {len(all_judge_results)} documents')

In [ ]:
# ── Judge results summary table ───────────────────────────────────────────────
judge_rows = []
for r in all_judge_results:
    judge_rows.append({
        'File': r.file_name,
        'Type': r.file_type.upper(),
        'Completeness': f'{r.completeness_score:.0f}',
        'Structure': f'{r.structure_fidelity_score:.0f}',
        'Noise': f'{r.noise_hallucination_score:.0f}',
        'Arabic': f'{r.arabic_fidelity_score:.0f}',
        'Overall': f'{r.overall_score:.1f}',
        'Verdict': f'{r.verdict_emoji} {r.verdict}',
        'Chunks': r.chunks_evaluated,
        'Fallback?': '✓' if r.used_fallback_heuristic else '✗',
    })

display(pd.DataFrame(judge_rows).style.set_caption('Gemini LLM-as-a-Judge Results'))

In [ ]:
# ── Display detailed metrics + judge card for each document ──────────────────
from evals.extraction.metrics import ExtractionMetrics

metrics_by_stem = {m.file_name: m for m in all_metrics}
judge_by_stem   = {r.file_name: r for r in all_judge_results}

for stem in extracted_docs:
    if stem not in metrics_by_stem:
        continue
    m = metrics_by_stem[stem]
    jr = judge_by_stem.get(stem)
    reviewer.display_metrics_summary(m.to_dict(), judge_result=jr)

## 👤 Section 7 — Human-In-The-Loop (HITL) Review

Interactive side-by-side diff between extracted text and ground truth.
Review documents individually — especially those with low scores or FAIL verdicts.

In [ ]:
# ── Auto-identify documents needing review (score < 80 or FAIL/CONDITIONAL_PASS) ──
review_queue = [
    r for r in all_judge_results
    if r.verdict in ('FAIL', 'CONDITIONAL_PASS') or r.overall_score < 80
]

print(f'Documents needing HITL review: {len(review_queue)}')
for r in review_queue:
    print(f'  {r.verdict_emoji} {r.file_name} [{r.file_type}] — Score: {r.overall_score:.1f}')

In [ ]:
# ── Display diff for each document requiring review ───────────────────────────
for r in review_queue:
    stem = r.file_name
    if stem not in extracted_docs or stem not in ground_truths:
        continue
    
    doc = extracted_docs[stem]
    gt  = ground_truths[stem]
    
    reviewer.display_diff(
        extracted_text=doc.raw_text,
        ground_truth_text=gt['body'],
        title=f'{r.verdict_emoji} [{r.file_type.upper()}] {stem}  |  Score: {r.overall_score:.1f}/100'
    )
    
    # Show specific missing elements
    if r.missing_critical_elements:
        from IPython.display import display, HTML
        missing_html = ''.join(f'<li><code>{e}</code></li>' for e in r.missing_critical_elements)
        display(HTML(f"""
        <div style='background:#fef2f2;border-left:4px solid #ef4444;padding:12px;
                    border-radius:4px;margin:4px 0;font-family:sans-serif'>
          <strong style='color:#991b1b'>⚠️ Missing Critical Elements:</strong>
          <ul style='margin:6px 0 0 16px;color:#7f1d1d'>{missing_html}</ul>
        </div>"""))

In [ ]:
# ── Record your HITL review decisions ────────────────────────────────────────
# Customize action, human_score, and comment as needed per document

for r in all_judge_results:
    action = 'APPROVE' if r.overall_score >= 80 else 'FLAG'
    reviewer.record_review(
        file_name=r.file_name,
        file_type=r.file_type,
        action=action,
        auto_score=r.overall_score,
        verdict=r.verdict,
        comment='Auto-reviewed' if r.overall_score >= 80 else f'Missing: {r.missing_critical_elements[:2]}',
        flagged_issues=r.missing_critical_elements,
    )

## 📈 Section 8 — Aggregated Benchmark Report

### Micro vs. Macro Aggregation + Per-Document-Type Breakdown

In [ ]:
from evals.extraction.metrics import aggregate_per_type

aggregations = aggregate_per_type(all_metrics)

print('\n' + '='*80)
print('EXTRACTION QUALITY BENCHMARK — AGGREGATED RESULTS')
print('='*80)

agg_rows = []
for label, agg in aggregations.items():
    print(f'\n📊 {label.upper()} ({agg.n_documents} docs)')
    print(f'   Macro Word F1:    {agg.macro_word_f1:.1%}')
    print(f'   Macro ROUGE-L:    {agg.macro_rouge_l:.1%}')
    print(f'   Micro Word F1:    {agg.micro_word_f1:.1%}')
    print(f'   Macro Entity Ret: {agg.macro_entity_recall:.1%}')
    print(f'   Macro Order τ:    {agg.macro_section_order_tau:.3f}')
    print(f'   Macro Table Ret:  {agg.macro_table_retention:.1%}')
    print(f'   Macro Media Ret:  {agg.macro_media_retention:.1%}')
    
    agg_rows.append({
        'Scope': label,
        'N': agg.n_documents,
        'Macro Word F1': f'{agg.macro_word_f1:.1%}',
        'Macro ROUGE-L': f'{agg.macro_rouge_l:.1%}',
        'Micro Word F1': f'{agg.micro_word_f1:.1%}',
        'Macro Entity': f'{agg.macro_entity_recall:.1%}',
        'Macro Order τ': f'{agg.macro_section_order_tau:.3f}',
        'Macro Table': f'{agg.macro_table_retention:.1%}',
        'Macro Media': f'{agg.macro_media_retention:.1%}',
    })

display(pd.DataFrame(agg_rows).style.set_caption('Micro/Macro + Per-Type Aggregated Metrics'))

In [ ]:
# ── Gemini Judge aggregated statistics ───────────────────────────────────────
from collections import Counter

verdict_counts = Counter(r.verdict for r in all_judge_results)
avg_overall = sum(r.overall_score for r in all_judge_results) / len(all_judge_results) if all_judge_results else 0

print('\nGemini Judge Summary')
print('─' * 40)
print(f'  ✅ PASS:              {verdict_counts["PASS"]}')
print(f'  ⚠️  CONDITIONAL_PASS: {verdict_counts["CONDITIONAL_PASS"]}')
print(f'  ❌ FAIL:              {verdict_counts["FAIL"]}')
print(f'  📊 Avg Overall Score: {avg_overall:.1f}/100')

# Per-type judge averages
from collections import defaultdict
type_judge: dict[str, list] = defaultdict(list)
for r in all_judge_results:
    type_judge[r.file_type].append(r.overall_score)

print('\nPer-Type Gemini Judge Scores:')
for ftype, scores in type_judge.items():
    print(f'  {ftype.upper()}: avg={sum(scores)/len(scores):.1f}/100 ({len(scores)} docs)')

In [ ]:
# ── Actionable recommendations across all documents ──────────────────────────
all_recs = []
for r in all_judge_results:
    for rec in r.recommendations:
        if rec and rec not in all_recs:
            all_recs.append(f'[{r.file_name} / {r.file_type}] {rec}')

if all_recs:
    print('\n🔧 Actionable Extractor Recommendations:')
    print('─' * 60)
    for i, rec in enumerate(all_recs[:15], 1):
        print(f'  {i}. {rec}')

In [ ]:
# ── Export full benchmark report ─────────────────────────────────────────────
from evals.extraction.hitl import save_benchmark_report

json_path, md_path = save_benchmark_report(
    all_metrics=[m.to_dict() for m in all_metrics],
    aggregations=aggregations,
    judge_results=[r.to_dict() for r in all_judge_results],
    report_dir=str(REPORTS_DIR),
)

# Save HITL audit log
reviewer.save_audit_log('extraction_hitl_audit_log.json')

print('\n✅ Evaluation complete!')
print(f'📊 JSON report: {json_path}')
print(f'📄 MD summary:  {md_path}')